# Week 6 Homework: Prompt Engineering Audit

## Task: turn a provider's price-list flyer into a web price sheet

At MyTeachers123 I help licensed family child care providers in California advertise their openings.
For each provider I make a price sheet: license number, program name, prices by age, address,
phone numbers, and a QR code that links to `https://myteachers123.com/<license number>`.
Until now these were images made by hand. Now I need them as **HTML web pages**, so every week I
turn a flyer (or the information a provider sends me) into structured data, and generate the page
from that data.

The LLM step is the hard part: reading the flyer and producing **validated JSON**:

- `license_number`, `program_name`, `program_name_alt` (second-language name)
- `price_unit` (month / week / day / hour) and `prices`: label, age range **converted to months**, amount
- `street`, `city`, `zip`, `phones` (number + language)

The HTML page and QR code are then generated by code (no LLM), so a wrong number in the JSON
becomes a wrong price on a public page. That is why correctness and validation matter.

**Test inputs:** five real price-list flyers from my work (`data/flyers/f1.jpg` ... `f5.jpg`).
Names written in Chinese are stored as `\uXXXX` escapes in `data/inputs.json` and are printed in
escaped form in this notebook.

1. Part 1: five prompt variations x five flyers, scoring, results table
2. Part 2: failure analysis on the worst variation, three adversarial flyers, three iterations, final prompt V6
3. Part 3: LangChain chain with `PromptTemplate`, `ChatOpenAI`, `PydanticOutputParser`, `OutputFixingParser`, and HTML output

In [ ]:
# Setup. In Colab: add OPENAI_API_KEY in Colab Secrets (key icon), then Runtime > Run all.
%pip install -q "openai>=3.42" "langchain-core>=1.7.3" "langchain-openai>=1.7.3" "langchain-classic>=1.0.8" "pydantic>=2.14" "tiktoken>=0.14" "pandas>=3.0" "pillow>=12" "segno>=1.6.6" "jinja2>=3.1"

In [ ]:
import base64, io, json, math, os, re, time
from pathlib import Path
from typing import Literal

import pandas as pd
from PIL import Image
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

try:  # Colab secret
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY") or "")
except Exception:
    pass

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
PRICE_IN, PRICE_OUT = 0.75, 4.50          # USD per 1M tokens, OpenAI list price for gpt-5.4-mini (checked 2026-11-01)
IMAGE_WIDTH = 768                          # flyers are resized before sending (fewer image tokens, still readable)
QR_BASE = "https://myteachers123.com/"
# MOCK = True runs everything offline with canned answers, only to check the code.
MOCK = os.getenv("MOCK", "0") == "1" or not os.getenv("OPENAI_API_KEY")
HERE = Path(".") if Path("data").exists() else Path("genai-course/week6/homework")
DATA = HERE / "data"
pd.set_option("display.max_colwidth", 120)
print("model:", MODEL, "| mock mode:", MOCK)


def asc(x) -> str:
    # print non-ASCII text (e.g. Chinese names) as \u escapes
    s = x if isinstance(x, str) else json.dumps(x, ensure_ascii=False)
    return s.encode("ascii", "backslashreplace").decode()

In [ ]:
import tiktoken

try:
    ENC = tiktoken.encoding_for_model(MODEL)
except Exception:
    try:
        ENC = tiktoken.get_encoding("o200k_base")   # GPT-5 family tokenizer
    except Exception:
        ENC = None
        print("tiktoken encoding unavailable; using len/4")


def n_tokens(text: str) -> int:
    return len(ENC.encode(text)) if ENC else max(1, len(text) // 4)


def text_tokens(messages) -> int:
    # tiktoken counts the TEXT parts; images are counted separately (see image_tokens_estimate)
    total = 3
    for m in messages:
        parts = m["content"] if isinstance(m["content"], list) else [{"type": "text", "text": m["content"]}]
        total += 3 + sum(n_tokens(p["text"]) for p in parts if p["type"] == "text")
    return total


def image_data_url(path: Path, width: int = IMAGE_WIDTH) -> str:
    im = Image.open(path).convert("RGB")
    im = im.resize((width, round(width * im.height / im.width)))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=85)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def image_tokens_estimate(n_images: int) -> int:
    # Estimate only (used in MOCK mode): 32-px patches x 1.62 for mini models. Real runs use the API's counts.
    w, h = IMAGE_WIDTH, round(IMAGE_WIDTH * 11 / 8.5)
    return n_images * round(math.ceil(w / 32) * math.ceil(h / 32) * 1.62)

## Test inputs (5 real flyers) and expected answers

`gold` is my hand-checked answer for each flyer. Age ranges follow one rule, which is also what the
website needs to find the right price for a child's age:

| Printed | Months |
|---|---|
| `A - B months` | A to B |
| `A - B years` | A x 12 to (B + 1) x 12 - 1 (e.g. 2 - 4 years = 24 to 59) |
| `N years old +` | N x 12 to open-ended |
| no age (e.g. "After School") | null, null |

In [ ]:
INPUTS = json.loads((DATA / "inputs.json").read_text(encoding="utf-8"))
EXAMPLES = json.loads((DATA / "examples" / "examples.json").read_text(encoding="utf-8"))
pd.DataFrame([{"id": x["id"], "file": x["image"], "license": x["gold"]["license_number"],
               "program": asc(x["gold"]["program_name"]), "price lines": len(x["gold"]["prices"]),
               "what makes it tricky": x["note_en"]} for x in INPUTS])

In [ ]:
# Thumbnails of the five real flyers
from IPython.display import display
sheet = Image.new("RGB", (5 * 220, 285), "white")
for i, x in enumerate(INPUTS):
    t = Image.open(DATA / x["image"]).convert("RGB"); t.thumbnail((220, 285)); sheet.paste(t, (220 * i, 0))
display(sheet)

In [ ]:
Unit = Literal["month", "week", "day", "hour"]


class PriceLine(BaseModel):
    label: str
    age_min_months: int | None = None
    age_max_months: int | None = None
    amount: float
    unit: Unit | None = None          # only when this line's unit differs from price_unit


class PhoneLine(BaseModel):
    number: str
    language: str | None = None


class PriceSheet(BaseModel):
    # Part 1 format check: structure and types only (business rules come in Part 3)
    license_number: str
    program_name: str
    program_name_alt: str | None = None
    price_unit: Unit | None
    prices: list[PriceLine]
    street: str | None = None
    city: str | None = None
    zip: str | None = None
    phones: list[PhoneLine] = Field(default_factory=list)
    needs_clarification: str | None = None


SCHEMA_JSON = json.dumps(PriceSheet.model_json_schema())

## Part 1: five prompt variations

| # | Strategy | What is different |
|---|---|---|
| V1 | Zero-shot | One sentence + the image. No field list, no rules, no examples |
| V2 | Few-shot | Two example flyers (fictional, made with my HTML template) with their JSON, then the test flyer |
| V3 | Chain-of-thought | Transcribe every line, then reason about names, units, age conversion and phones, then `FINAL:` JSON |
| V4 | Role + structured output | Persona system prompt + the Pydantic JSON schema; reply validated by Pydantic |
| V5 | Hybrid | Persona + explicit rules (age conversion, units, nulls, names, phones) + one text-only example + schema; no visible reasoning |

In [ ]:
AGE_RULES = """Age ranges in months: "A - B months" -> A to B; "A - B years" -> A*12 to (B+1)*12-1;
"N years old +" -> N*12 to null; "A weeks" -> whole months rounded down; a label with no age (e.g. "After School") -> null, null."""

V4_SYSTEM = ("You are a data entry specialist at MyTeachers123. You turn California family child care providers' "
             "price-list flyers into exact data for their web price sheets. Accuracy matters: these prices are published.")

V5_RULES = f"""Rules:
- license_number: the 9 digits after "LIC #".
- program_name: the main program name exactly as printed (keep Chinese or other scripts as they are).
  program_name_alt: a second name in another language or a slogan printed next to it, else null.
- price_unit: from the title, e.g. "Price List (monthly)" -> month. If the flyer never says the unit, use null
  and ask in needs_clarification. Never assume monthly.
- prices: one item per printed line, in order; label exactly as printed; amount as a number.
  If a line shows its own unit (e.g. "$25/hour") that differs from price_unit, put it in that line's unit.
- {AGE_RULES}
- street, city, zip, phones: only what you can read on the flyer; null or [] if not visible. Do not use outside knowledge.
- phones: format 626.509.0788; language in English (e.g. Chinese, English) if printed next to the number, else null.
- Output only the JSON object."""


def img(item):
    return {"type": "image_url", "image_url": {"url": image_data_url(DATA / item["image"])}}


def v1_zero_shot(item):
    return [{"role": "user", "content": [{"type": "text", "text": "Extract the information on this child care price list flyer as JSON."}, img(item)]}]


def v2_few_shot(item):
    msgs = [{"role": "system", "content": "Convert child care price-list flyers to JSON exactly like the examples."}]
    for ex in EXAMPLES:
        msgs.append({"role": "user", "content": [{"type": "text", "text": "Flyer:"}, img(ex)]})
        msgs.append({"role": "assistant", "content": json.dumps(ex["json"])})
    msgs.append({"role": "user", "content": [{"type": "text", "text": "Flyer:"}, img(item)]})
    return msgs


def v3_cot(item):
    return [{"role": "user", "content": [{"type": "text", "text":
        "This is a family child care price-list flyer. Think step by step before answering:\n"
        "1. Transcribe every line of text exactly as printed.\n"
        "2. Find the license number, the program name, and any second-language name.\n"
        "3. Find the price unit from the title (monthly, weekly, ...). Check each line for a different unit.\n"
        "4. For each price line, convert the age range to months (min and max).\n"
        "5. Find the address and phone numbers with any language label.\n"
        "Then write FINAL: followed by one JSON object with keys license_number, program_name, program_name_alt, "
        "price_unit, prices (label, age_min_months, age_max_months, amount, unit), street, city, zip, "
        "phones (number, language), needs_clarification."}, img(item)]}]


def v4_role_structured(item):
    return [{"role": "system", "content": V4_SYSTEM},
            {"role": "user", "content": [{"type": "text", "text":
                f"Return only a JSON object that validates against this JSON schema:\n{SCHEMA_JSON}"}, img(item)]}]


def v5_hybrid(item):
    example = json.dumps(EXAMPLES[1]["json"])
    return [{"role": "system", "content": V4_SYSTEM + "\n\n" + V5_RULES},
            {"role": "user", "content": [{"type": "text", "text":
                f"Example output for another flyer (mixed units):\n{example}\n\nJSON schema:\n{SCHEMA_JSON}\n\n"
                "Now the flyer:"}, img(item)]}]


VARIATIONS = {"V1 zero-shot": v1_zero_shot, "V2 few-shot": v2_few_shot, "V3 chain-of-thought": v3_cot,
              "V4 role + structured": v4_role_structured, "V5 hybrid": v5_hybrid}

In [ ]:
from openai import OpenAI

client = None if MOCK else OpenAI()


def _mock_reply(messages, item):
    # Offline stand-in that returns the expected answer (V1 gets a broken key to exercise the scoring code)
    body = json.dumps(item["gold"])
    first = next(p["text"] for p in messages[-1]["content"] if p["type"] == "text")
    if first.startswith("This is a family"):
        return "1. ...\nFINAL: " + body
    if first.startswith("Extract the information"):
        return "```json\n" + body.replace('"license_number"', '"license"') + "\n```"
    return body


def call(messages, item):
    t0 = time.time()
    n_img = sum(1 for m in messages if isinstance(m["content"], list) for p in m["content"] if p["type"] == "image_url")
    if MOCK:
        text = _mock_reply(messages, item)
        api_in, api_out = None, None
    else:
        r = client.chat.completions.create(model=MODEL, messages=messages)
        text = r.choices[0].message.content or ""
        api_in, api_out = r.usage.prompt_tokens, r.usage.completion_tokens
    tok_text, tok_out = text_tokens(messages), n_tokens(text)
    tok_in = api_in if api_in is not None else tok_text + image_tokens_estimate(n_img)
    return {"text": text, "images": n_img, "tok_text_in": tok_text, "tok_in": tok_in,
            "tok_out": api_out if api_out is not None else tok_out, "seconds": round(time.time() - t0, 2)}


def extract_json(text: str):
    if "FINAL:" in text:
        text = text.split("FINAL:")[-1]
    m = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.S)
    if m:
        return json.loads(m.group(1))
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        raise ValueError("no JSON object")
    return json.loads(m.group(0))

In [ ]:
RUNS = []
for item in INPUTS:
    for name, build in VARIATIONS.items():
        RUNS.append({"input": item["id"], "variation": name, **call(build(item), item)})
print(len(RUNS), "calls done")

### Scoring

- **Format compliance (pass/fail):** the reply contains JSON that validates against `PriceSheet`.
- **Correctness (1-5), automatic, against my expected answer**, 5 points:
  license number 1, names 1 (main 0.5 + second name 0.5), prices 2 (F1 over amount + age range + effective unit),
  address 0.5, phones 0.5 (numbers and language labels). Points map to `1 + round(points * 4 / 5)`.
  A reply that fails the format check scores 1. I checked every automatic score; overrides go in `CORRECTNESS_OVERRIDE`.
- **Usefulness to me (1-5):** my judgment: could I generate the web page from this JSON without fixing anything?
- **Tokens:** text tokens counted with tiktoken; image tokens come from the API's usage counts (estimated in MOCK mode).
  Cost per 1K calls uses input and output tokens at the gpt-5.4-mini list price.

In [ ]:
def parse(text):
    try:
        return PriceSheet.model_validate(extract_json(text)), None
    except Exception as e:
        return None, f"{type(e).__name__}: {str(e)[:120]}"


def norm(s):
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip() if s is not None else None


def digits(s):
    return re.sub(r"\D", "", s or "")


def f1(got, want):
    if not got and not want:
        return 1.0
    tp = len(set(got) & set(want))
    return 0.0 if tp == 0 else 2 * tp / (len(set(got)) + len(set(want)))


def price_keys(sheet: dict):
    return [(round(float(p["amount"]), 2), p.get("age_min_months"), p.get("age_max_months"),
             p.get("unit") or sheet.get("price_unit")) for p in sheet["prices"]]


def correctness(obj, gold):
    if obj is None:
        return 1, 0.0
    d = obj.model_dump()
    pts = 1.0 if digits(d["license_number"]) == gold["license_number"] else 0.0
    pts += 0.5 * (norm(d["program_name"]) == norm(gold["program_name"]))
    pts += 0.5 * (norm(d["program_name_alt"]) == norm(gold["program_name_alt"]))
    pts += 2 * f1(price_keys(d), price_keys(gold))
    pts += 0.5 * all(norm(d[k]) == norm(gold[k]) for k in ("street", "city", "zip"))
    phones = lambda s: sorted((digits(p["number"]), (p.get("language") or "").lower()) for p in s["phones"])
    pts += 0.5 * (phones(d) == phones(gold))
    return 1 + round(pts * 4 / 5), round(pts, 2)


GOLD = {x["id"]: x["gold"] for x in INPUTS}
for r in RUNS:
    obj, err = parse(r["text"])
    r["format_pass"], r["parse_error"] = obj is not None, err
    r["correctness"], r["points"] = correctness(obj, GOLD[r["input"]])

pd.DataFrame(RUNS)[["input", "variation", "format_pass", "correctness", "points", "images", "tok_in", "tok_out", "parse_error"]]

In [ ]:
# My manual review. Keys: (input id, variation name).
CORRECTNESS_OVERRIDE = {
    # ("f1", "V1 zero-shot"): 2,
}
# Usefulness to me, 1-5 (5 = I can publish the page from this JSON as is). TODO: fill in after reading the replies below.
USEFULNESS = {
    # ("f1", "V5 hybrid"): 5,
}
for r in RUNS:
    k = (r["input"], r["variation"])
    r["correctness"] = CORRECTNESS_OVERRIDE.get(k, r["correctness"])
    r["usefulness"] = USEFULNESS.get(k)
print("usefulness scores still missing:", sum(r["usefulness"] is None for r in RUNS), "of", len(RUNS))

In [ ]:
# Raw replies (read these before scoring usefulness)
for r in RUNS:
    print(f"=== {r['input']} | {r['variation']} | format={'PASS' if r['format_pass'] else 'FAIL'} | correctness={r['correctness']}")
    print(asc(r["text"][:900]), "\n")

In [ ]:
def results_table(runs):
    df = pd.DataFrame(runs)
    g = df.groupby("variation", sort=False)
    tin, tout = g["tok_in"].mean(), g["tok_out"].mean()
    out = pd.DataFrame({
        "Avg Correctness": g["correctness"].mean().round(2),
        "Format Pass Rate": (g["format_pass"].mean() * 100).round(0).astype(int).astype(str) + "%",
        "Avg Usefulness": g["usefulness"].apply(lambda s: round(s.dropna().mean(), 2) if s.notna().any() else None),
        "Avg Tokens": (tin + tout).round(0).astype(int),
        "Text tokens in (tiktoken)": g["tok_text_in"].mean().round(0).astype(int),
        "Images per call": g["images"].mean().round(0).astype(int),
        "Cost/1K calls ($)": ((tin * PRICE_IN + tout * PRICE_OUT) / 1e6 * 1000).round(2),
    })
    out.index.name = "Variation"
    return out


TABLE1 = results_table(RUNS)
TABLE1

In [ ]:
def pick_winner(t):
    t = t.assign(_fmt=t["Format Pass Rate"].str.rstrip("%").astype(int), _use=t["Avg Usefulness"].fillna(0))
    return t.sort_values(["Avg Correctness", "_fmt", "_use", "Cost/1K calls ($)"],
                         ascending=[False, False, False, True]).index[0]


WINNER = pick_winner(TABLE1)
w = TABLE1.loc[WINNER]
print(f"Winner: {WINNER} (correctness {w['Avg Correctness']}, format {w['Format Pass Rate']}, ${w['Cost/1K calls ($)']}/1K calls)")

## Part 2: failure analysis and iteration

I run the worst variation from Part 1 on three flyers I designed to break it. Each one is a real
flyer edited to target one failure mode:

| id | Edit | Failure mode targeted |
|---|---|---|
| a1 | "(monthly)" removed from the title, so no unit appears anywhere | Ambiguous instruction |
| a2 | dim, blurred, tilted phone photo with the address and phone cut off | Missing context (and hallucination) |
| a3 | one line changed to "$25/hour" on a "monthly" sheet | Conflicting constraints |

Taxonomy used: ambiguous instruction, missing format specification, conflicting constraints,
missing context, hallucination / over-generation. (To check against Slide 10.)

In [ ]:
WORST = TABLE1.assign(_fmt=TABLE1["Format Pass Rate"].str.rstrip("%").astype(int)) \
              .sort_values(["Avg Correctness", "_fmt"]).index[0]
print("Worst variation:", WORST)
ADV = json.loads((DATA / "adversarial.json").read_text(encoding="utf-8"))
sheet = Image.new("RGB", (3 * 220, 285), "white")
for i, a in enumerate(ADV):
    t = Image.open(DATA / a["image"]).convert("RGB"); t.thumbnail((220, 285)); sheet.paste(t, (220 * i, 0))
display(sheet)
pd.DataFrame([{"id": a["id"], "failure mode": a["failure_mode"], "design": a["design"]} for a in ADV])

In [ ]:
def compare(obj, gold):
    # field-by-field differences between the model's answer and the expected one
    if obj is None:
        return "no valid JSON"
    d = obj.model_dump()
    diffs = []
    for k in ("license_number", "program_name", "program_name_alt", "price_unit", "street", "city", "zip"):
        if norm(str(d[k]) if d[k] is not None else None) != norm(str(gold[k]) if gold[k] is not None else None):
            diffs.append(f"{k}: expected {gold[k]!r}, got {d[k]!r}")
    if sorted(price_keys(d), key=str) != sorted(price_keys(gold), key=str):
        diffs.append(f"prices: expected {price_keys(gold)}, got {price_keys(d)}")
    if [digits(p["number"]) for p in d["phones"]] != [digits(p["number"]) for p in gold["phones"]]:
        diffs.append(f"phones: expected {[p['number'] for p in gold['phones']]}, got {[p['number'] for p in d['phones']]}")
    return "; ".join(diffs) or "matches"


ADV_RUNS = []
for a in ADV:
    r = call(VARIATIONS[WORST](a), a)
    obj, err = parse(r["text"])
    score, _ = correctness(obj, a["gold"])
    ADV_RUNS.append({"id": a["id"], "failure mode": a["failure_mode"], "format_pass": obj is not None,
                     "correctness": score, "expected vs actual": asc(compare(obj, a["gold"]))})
pd.DataFrame(ADV_RUNS)

### Three iterations on the worst prompt

| Iteration | Change | Failure mode addressed |
|---|---|---|
| 1 | Field list + "if the unit is not printed, use null and ask; never assume monthly" | Ambiguous instruction |
| 2 | "Only report text you can read; null if not visible; no outside knowledge" | Missing context / hallucination |
| 3 | Line-level units, age conversion rules, phone format, and the exact JSON schema | Conflicting constraints + missing format |

In [ ]:
FIELDS = ("Fields: license_number, program_name, program_name_alt, price_unit (month/week/day/hour), "
          "prices (label, age_min_months, age_max_months, amount, unit), street, city, zip, phones (number, language), "
          "needs_clarification.")
R1 = "If the flyer does not print the price unit, set price_unit to null and ask in needs_clarification. Never assume monthly."
R2 = ("Only report text you can actually read on the image. If the address or phone numbers are not visible, use null "
      "or an empty list. Do not use outside knowledge to fill anything in.")
R3 = ("price_unit is the unit in the title. If a line prints a different unit (e.g. '$25/hour'), set that line's unit. "
      + AGE_RULES + " Phones as 626.509.0788. Return only JSON valid against this schema:\n" + SCHEMA_JSON)


def _add(messages, extra):
    m = [dict(x) for x in messages]
    parts = [dict(p) for p in m[-1]["content"]]
    parts[0] = {"type": "text", "text": parts[0]["text"] + "\n" + extra}
    m[-1]["content"] = parts
    return m


def iter1(item):
    return _add(VARIATIONS[WORST](item), FIELDS + "\n" + R1)


def iter2(item):
    return _add(iter1(item), R2)


def iter3(item):
    return _add(iter2(item), R3)


ITERS = {"original": VARIATIONS[WORST], "iteration 1": iter1, "iteration 2": iter2, "iteration 3 = V6": iter3}
ITER_RUNS = []
for name, build in ITERS.items():
    for item in ADV + INPUTS:
        r = call(build(item), item)
        obj, err = parse(r["text"])
        score, _ = correctness(obj, item["gold"])
        ITER_RUNS.append({"version": name, "input": item["id"], "set": "adversarial" if item["id"].startswith("a") else "original",
                          "format_pass": obj is not None, "correctness": score, "tokens": r["tok_in"] + r["tok_out"],
                          "diff": asc(compare(obj, item["gold"]))})
it = pd.DataFrame(ITER_RUNS)
it.pivot_table(index="version", columns="set", values=["correctness", "format_pass"], aggfunc="mean", sort=False).round(2)

In [ ]:
# Per-input detail for the adversarial flyers across iterations
it[it["set"] == "adversarial"][["version", "input", "format_pass", "correctness", "diff"]]

In [ ]:
# The final, sixth prompt variant (V6) as sent for flyer a1 (image omitted)
for m in iter3(ADV[0]):
    parts = m["content"] if isinstance(m["content"], list) else [{"type": "text", "text": m["content"]}]
    print(f"[{m['role']}]")
    for p in parts:
        print(asc(p["text"]) if p["type"] == "text" else "<image>")

### Error analysis: license numbers that cannot be verified

A correct extraction is not the same as a correct price sheet. I checked every extracted license number
against CDSS on 2026-11-01:

- The CDSS Care Facility Search API returned **503 Service Unavailable** for all five numbers.
- The CHHS open data (Family Child Care Homes and Child Care Centers, last updated 2025-05-25) has **none** of them.
- A third-party mirror of CCLD records (ReadyRule, refreshed 2026-11-02) confirms only **198021766**
  (Wang Hua Family Child Care, West Covina, capacity 14, licensed 2025-07-25).

So 4 of 5 flyers carry a license number nobody can confirm today. The model read all of them "correctly",
but no prompt can tell whether a printed number is real, current, or mistyped by the provider: the
model has no access to the licensing database, and asking it would invite a confident hallucination.

| Failure | Class | Fix |
|---|---|---|
| Unverifiable license number | Missing context (outside the prompt and the model) | Look up CDSS in code; publish only `verified` sheets; human check otherwise |
| Official name differs from flyer name (198021766: CDSS "Wang Hua Family Child Care" vs flyer program name) | Not an error: business name vs licensee name | Keep both: `program_name` for the page, CDSS name for verification |

In [ ]:
CHECK = json.loads((DATA / "license_check.json").read_text())
pd.DataFrame(CHECK["results"])[["flyer", "license", "status", "source"]]

## Part 3: LangChain implementation of the winning variation (V5 hybrid)

- `PromptTemplate` holds the instructions, with variables `brand` and `age_rules`
  (plus `format_instructions` as a partial); it becomes the system message of a `ChatPromptTemplate`
  whose human message carries `received_on` and the flyer `image_url`.
- `ChatOpenAI` (`gpt-5.4-mini`, vision input)
- `PydanticOutputParser` on `WebPriceSheet`: typed fields, `field_validator`s and `model_validator`s
- `OutputFixingParser`: when parsing or validation fails, the bad output and the error go back to the model to fix
- Output: an HTML price sheet with a QR code for each flyer, written to `output/`

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_classic.output_parsers import OutputFixingParser
from langchain_openai import ChatOpenAI

RATE_BOUNDS = {"hour": (5, 150), "day": (20, 500), "week": (50, 2000), "month": (200, 8000)}


class WebPriceLine(BaseModel):
    label: str
    age_min_months: int | None = Field(None, ge=0, le=216)
    age_max_months: int | None = Field(None, ge=0, le=216)
    amount: float
    unit: Unit | None = None

    @model_validator(mode="after")
    def age_order(self):
        if self.age_min_months is not None and self.age_max_months is not None and self.age_max_months < self.age_min_months:
            raise ValueError(f"'{self.label}': age_max_months {self.age_max_months} is below age_min_months "
                             f"{self.age_min_months}; ages must be in months")
        return self


class WebPriceSheet(BaseModel):
    license_number: str
    program_name: str
    program_name_alt: str | None = None
    price_unit: Unit | None
    prices: list[WebPriceLine] = Field(min_length=1)
    street: str | None = None
    city: str | None = None
    zip: str | None = None
    phones: list[PhoneLine] = Field(default_factory=list)
    needs_clarification: str | None = None

    @field_validator("license_number")
    @classmethod
    def nine_digits(cls, v):
        d = re.sub(r"\D", "", v)
        if len(d) != 9:
            raise ValueError(f"license number '{v}' must have 9 digits (CDSS facility number)")
        return d

    @field_validator("zip")
    @classmethod
    def zip5(cls, v):
        if v is not None and not re.fullmatch(r"\d{5}", v):
            raise ValueError(f"zip '{v}' must be 5 digits")
        return v

    @field_validator("phones")
    @classmethod
    def phone_format(cls, v):
        for p in v:
            d = re.sub(r"\D", "", p.number)
            if len(d) != 10:
                raise ValueError(f"phone '{p.number}' must have 10 digits")
            p.number = f"{d[:3]}.{d[3:6]}.{d[6:]}"
        return v

    @model_validator(mode="after")
    def plausible_prices(self):
        for p in self.prices:
            unit = p.unit or self.price_unit
            if unit is None:
                continue
            lo, hi = RATE_BOUNDS[unit]
            if not lo <= p.amount <= hi:
                raise ValueError(f"'{p.label}': ${p.amount:g} per {unit} is outside ${lo}-${hi} "
                                 "(misread amount or wrong unit)")
        return self

    @property
    def qr_url(self):
        return QR_BASE + self.license_number


parser = PydanticOutputParser(pydantic_object=WebPriceSheet)

instructions = PromptTemplate(
    template=V4_SYSTEM.replace("MyTeachers123", "{brand}") + "\n\n" + V5_RULES.replace(AGE_RULES, "{age_rules}")
             + "\n\n{format_instructions}",
    input_variables=["brand", "age_rules"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)
prompt = ChatPromptTemplate.from_messages([
    ("system", instructions.template),
    ("human", [{"type": "text", "text": "Flyer received on {received_on}. Extract the web price sheet data."},
               {"type": "image_url", "image_url": {"url": "{image_url}"}}]),
]).partial(format_instructions=parser.get_format_instructions())
print("PromptTemplate variables:", instructions.input_variables)
print("Chat prompt variables:   ", prompt.input_variables)

In [ ]:
if MOCK:
    from langchain_core.language_models.fake_chat_models import FakeListChatModel
    llm = FakeListChatModel(responses=[json.dumps(x["gold"]) for x in INPUTS])
else:
    llm = ChatOpenAI(model=MODEL, max_retries=2, timeout=120)

fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm, max_retries=2)
chain = prompt | llm | fixing_parser

In [ ]:
# 1) Three outputs that json.loads accepts but the Pydantic validators reject
good = dict(GOLD["f2"])
BAD = {
    "'2 years old +' read as ages 24 to 2": {**good, "prices": [{**good["prices"][4], "age_min_months": 24, "age_max_months": 2}]},
    "'$3500' misread as 35 (per month)": {**good, "prices": [{**good["prices"][0], "amount": 35}]},
    "license number with a digit missing": {**good, "license_number": "19802171"},
}
for name, d in BAD.items():
    text = json.dumps(d)
    ok_json = bool(json.loads(text))
    try:
        parser.parse(text); verdict = "accepted"
    except Exception as e:
        verdict = "REJECTED: " + str(e).split("Got: ")[-1].split("For further")[0].replace("\n", " ").strip()[:200]
    print(f"{name}\n  json.loads OK: {ok_json}\n  Pydantic: {asc(verdict)}\n")

In [ ]:
# 2) OutputFixingParser: a malformed reply is sent back to the model with the error and fixed
broken = "LIC 198021713, Li Family Child Care, monthly: 0-3 months 3500, 2 years old + 2600, phone 626 766 0278"
try:
    parser.parse(broken)
except Exception as e:
    print("plain parser fails:", type(e).__name__)
fixed = fixing_parser.parse(broken)
print("after OutputFixingParser:", asc(fixed.model_dump_json()[:300]))

In [ ]:
# 3) Run the chain on all 5 real flyers
CHAIN = {}
rows = []
for item in INPUTS:
    res = chain.invoke({"brand": "MyTeachers123", "age_rules": AGE_RULES, "received_on": item["received_on"],
                        "image_url": image_data_url(DATA / item["image"])})
    CHAIN[item["id"]] = res
    rows.append({"input": item["id"], "valid WebPriceSheet": isinstance(res, WebPriceSheet),
                 "license": res.license_number, "program": asc(res.program_name), "unit": res.price_unit,
                 "price lines": len(res.prices), "phones": len(res.phones), "qr_url": res.qr_url,
                 "CDSS verified": next((c["status"] for c in CHECK["results"] if c["license"] == res.license_number), "not checked"),
                 "correctness vs gold": correctness(PriceSheet.model_validate(res.model_dump()), GOLD[item["id"]])[0]})
assert all(r["valid WebPriceSheet"] for r in rows)
# Valid JSON is not enough to publish: only CDSS-verified licenses go live; the rest wait for a human check.
pd.DataFrame(rows)

### HTML price sheets with QR codes

The validated objects are rendered with the same template my capstone chatbot uses. Non-ASCII text
(Chinese names) is written as HTML character references, so the files are plain ASCII.

In [ ]:
FLYER_TEMPLATE = r'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{{ name }} | Price List</title>
<meta name="description" content="{{ name }}{% if name_alt %} | {{ name_alt }}{% endif %}, licensed family child care in {{ city }}, CA {{ zip }}. CDSS license #{{ license }}. {{ title }}.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Gaegu:wght@400;700&family=Itim&display=swap" rel="stylesheet">
<style>
  :root { --cream:#fcf8f3; --salmon:#e0907f; --ink:#6f7271; --yellow:#fdf0cb; }
  * { box-sizing:border-box; }
  html, body { margin:0; background:#ebe7e1; }
  body { font-family:"Gaegu", "Noto Sans SC", "Noto Sans TC", "PingFang SC", "Microsoft YaHei", sans-serif; }
  .toolbar { text-align:center; padding:10px; font:14px/1.4 -apple-system, "Segoe UI", Roboto, sans-serif; }
  .toolbar button, .toolbar a { margin:0 4px; padding:7px 14px; border-radius:8px; border:1px solid #c9c2b8;
    background:#fff; color:#333; text-decoration:none; cursor:pointer; font:inherit; }
  .page { position:relative; width:min(100vw, 8.5in); aspect-ratio:8.5 / 11; margin:0 auto 24px;
    background:var(--cream); overflow:hidden; container-type:inline-size; box-shadow:0 2px 14px rgba(0,0,0,.12); }
  .deco { position:absolute; inset:0; width:100%; height:100%; }
  .abs { position:absolute; }
  .brand { right:5.5%; top:15.5%; width:42%; text-align:center; color:var(--salmon); font-family:"Itim", sans-serif;
    font-size:4.6cqw; line-height:1.12; }
  .title { left:7.8%; top:26.5%; color:var(--salmon); font-weight:700; font-size:8.6cqw; line-height:1; }
  .prices { left:9.5%; top:36.5%; width:55%; color:var(--ink); font-size:5.1cqw; line-height:1.48; font-weight:700; }
  .prices.small { font-size:4.2cqw; line-height:1.42; }
  .prices div { white-space:nowrap; }
  .qr { left:66%; top:37.5%; width:27%; text-align:center; color:var(--ink); font-size:2.9cqw; font-weight:700; white-space:nowrap; }
  .qr .code { background:#fff; padding:2.5%; }
  .qr svg { display:block; width:100%; height:auto; }
  .enroll { left:9.5%; top:66.5%; color:var(--salmon); font-weight:700; font-size:8cqw; line-height:1; }
  .who { left:9.5%; top:74%; width:62%; color:var(--ink); font-weight:700; }
  .who .name { font-size:5.4cqw; line-height:1.15; }
  .who .name.long { font-size:4.3cqw; }
  .who .alt { font-size:5cqw; line-height:1.2; font-weight:400; }
  .who .addr { font-size:4.1cqw; line-height:1.3; margin-top:2cqw; }
  @media print {
    html, body { background:none; }
    .toolbar { display:none; }
    .page { width:8.5in; box-shadow:none; margin:0; }
    @page { size:letter; margin:0; }
  }
</style>
</head>
<body>
<div class="toolbar">
  <button type="button" onclick="window.print()">Print / Save as PDF</button>
</div>
<main class="page" aria-label="Price list">
  <svg class="deco" viewBox="0 0 850 1100" preserveAspectRatio="none" aria-hidden="true">
    <path d="M0,792 C38,700 150,690 230,688 C420,684 560,722 622,822 C700,962 680,1050 655,1100 L0,1100 Z" fill="#fdf0cb"/>
    <g fill="none" stroke-linecap="round" stroke-width="7">
      {% for x, y, c in [(38,18,'p'),(95,72,'p'),(2,92,'p'),(47,146,'p'),(138,138,'p'),(0,212,'p'),
                        (154,8,'b'),(210,52,'b'),(292,52,'b'),(242,116,'b'),(341,107,'b'),(434,99,'b'),(278,173,'b'),(390,36,'b'),(240,-6,'b'),
                        (566,890,'p'),(517,946,'p'),(616,935,'p'),(429,968,'p'),(478,1012,'p'),(578,1018,'p'),(671,995,'p'),(715,1062,'p'),(517,1078,'p'),(622,1078,'p'),
                        (770,753,'b'),(715,808,'b'),(820,797,'b'),(627,825,'b'),(682,869,'b'),(776,874,'b'),(720,935,'b'),(825,929,'b'),(759,995,'b')] %}
      <path d="M{{ x }},{{ y + 34 }} v-14 a17,17 0 0 1 34,0 v14" stroke="{{ '#e39a8b' if c == 'p' else '#cdd8de' }}"/>
      {% endfor %}
    </g>
    <g fill="none" stroke-linecap="butt" stroke-width="11">
      {% for r, col in [(98,'#e39a8b'),(86,'#cfd9df'),(74,'#eec4c2'),(62,'#d9ae5a'),(50,'#6e716f')] %}
      <path d="M{{ 655 - r }},165 V119 A{{ r }},{{ r }} 0 0 1 {{ 655 + r }},119 V165" stroke="{{ col }}"/>
      {% endfor %}
    </g>
  </svg>

  <div class="abs brand">{{ brand }}<br>LIC # {{ license }}</div>
  <h1 class="abs title" style="margin:0">{{ title }}</h1>
  <div class="abs prices{% if small %} small{% endif %}" style="font-size:{{ price_font }}cqw">
    {% for l in lines %}<div>{{ l.label }}: {{ l.price }}</div>{% endfor %}
  </div>
  <div class="abs qr">
    <a class="code" href="{{ qr_url }}" style="display:block" aria-label="Open {{ qr_url }}">{{ qr_svg|safe }}</a>
    <div>Scan for more details</div>
  </div>
  <div class="abs enroll">Now Enrolling for</div>
  <div class="abs who">
    <div class="name{% if name|length > 26 %} long{% endif %}">{{ name }}</div>
    {% if name_alt %}<div class="alt">{{ name_alt }}</div>{% endif %}
    <div class="addr">
      {% if street %}{{ street }}<br>{% endif %}{% if city or zip %}{{ city }}{% if city %}, {% endif %}CA {{ zip }}<br>{% endif %}
      {% for p in phones %}{{ p.number }}{% if p.language %} ({{ p.language }}){% endif %}<br>{% endfor %}
    </div>
  </div>
</main>
</body>
</html>
'''

In [ ]:
import segno
from jinja2 import Environment

UNIT_TITLE = {"month": "monthly", "week": "weekly", "day": "daily", "hour": "hourly"}
UNIT_SHORT = {"month": "/mo", "week": "/wk", "day": "/day", "hour": "/hr"}
_env = Environment(autoescape=True)


def qr_svg(url):
    buf = io.BytesIO()
    segno.make(url, error="m").save(buf, kind="svg", scale=1, border=2, xmldecl=False, omitsize=True, nl=False)
    return buf.getvalue().decode()


def render_html(s: WebPriceSheet) -> str:
    units = {p.unit or s.price_unit for p in s.prices}
    one = len(units) == 1 and None not in units
    title = f"Price List ({UNIT_TITLE[units.pop()]})" if one else "Price List"
    money = lambda a: f"${a:,.2f}".replace(".00", "").replace(",", "")
    lines = [{"label": p.label, "price": money(p.amount) + ("" if one or not (p.unit or s.price_unit) else UNIT_SHORT[p.unit or s.price_unit])}
             for p in s.prices]
    longest = max(len(l["label"]) + len(l["price"]) + 2 for l in lines)
    html = _env.from_string(FLYER_TEMPLATE).render(
        brand="MyTeachers123.com", license=s.license_number, title=title, lines=lines,
        name=s.program_name, name_alt=s.program_name_alt or "", street=s.street or "", city=s.city or "",
        zip=s.zip or "", phones=[p.model_dump() for p in s.phones], qr_url=s.qr_url, qr_svg=qr_svg(s.qr_url),
        small=len(lines) > 6, price_font=round(min(5.1, 54 / (0.53 * longest)), 2))
    return html.encode("ascii", "xmlcharrefreplace").decode()


OUT = HERE / "output"
OUT.mkdir(exist_ok=True)
for k, s in CHAIN.items():
    (OUT / f"price-sheet-{s.license_number}.html").write_text(render_html(s), encoding="ascii")
sorted(p.name for p in OUT.glob("*.html"))